# Epstein Files
The documents have been sources from a [GitHub repo](https://github.com/epstein-docs/epstein-docs.github.io/tree/main#) that has OCR'd documents using OpenAI.

The repo was last updated 4 months ago, so the files are not recent. This is something I will work on soon.

## Converting Into Parquet By Doc Types

In [1]:
import glob
import json
from collections import defaultdict
import pandas as pd
import duckdb
import datetime

In [70]:
all_files = glob.glob("./docs/**/*.json")

# Dictionary of [doc type] -> [file names]
doc_types = defaultdict(list[str])

for file_name in all_files:
    with open(file_name, "rt", encoding="utf-8") as file:
        data = json.load(file)

        metadata = data["document_metadata"]

        dt = metadata["document_type"].lower() if metadata["document_type"] is not None else "unknown"

        doc_types[dt].append(file_name)

In [77]:
# Getting document frequency and writing to csv for Tableu visuals

og_df = pd.DataFrame({
    "doc_type": [k for k in doc_types.keys()],
    "file_names": [v for v in doc_types.values()],
    "frequency": [len(v) for v in doc_types.values()]
})
og_df.to_csv("document_frequencies.csv", index=False)

In [80]:
new_df = pd.DataFrame(columns=["doc_type", "file_names", "frequency"])

In [ ]:
# These document types are too specific, i.e. 'court document' vs 'court transcript' vs 'court order'
target = "log"
targets = ["court", "email", "report", "myspace", "log"]

query = f"""
    WITH
        matches AS (
            SELECT
                doc_type,
                file_names,
                frequency
            FROM
                og_df
            WHERE
                regexp_matches(doc_type, '(?i)\\b{target}\\b')
        )
    SELECT
        '{target}_documents' AS doc_type,
        list_concat(list(file_names)) AS file_names,
        SUM(frequency) AS frequency
    FROM
        matches
"""
df = duckdb.query(query).df()

new_df = pd.concat([new_df, df])
# duckdb.query(query)

In [ ]:
## Writng CSV for parquet visualizations
(new_df.drop("file_names", axis=1)).to_csv("compressed_document_frequencies.csv", index=False)

### Making Parquet Files

In [174]:
DOC_METADATA = ["page_number", "document_number", "date", "document_type", "has_handwriting", "has_stamps"]
FULL_TEXT = ["full_text"]
ENTITIES = ["people", "organizations", "locations", "dates", "reference_numbers"]

ADD_NOTES = ["additional_notes"]

COLS = DOC_METADATA + FULL_TEXT + ENTITIES + ADD_NOTES

for _, (dt, fnames, freq) in new_df.iterrows():
    df = pd.DataFrame(columns=COLS)

    for file in fnames[0]:
        with open(file, "rt", encoding="utf-8") as f:
            data = json.load(f)
            try:
                '''
                Document Metadata has:
                    page_number : int
                    document_number : int
                    date : MM/DD/YYYY
                    document_type : str
                    has_handwriting : bool
                    has_stamps : bool
                '''
                metadata = data["document_metadata"]
                

                '''
                Full document text
                '''
                full_text = data["full_text"]

                '''
                Entities has
                    people : list[str]
                    organizations : list[str]
                    locations : list[str]
                    dates : list[strings] ----> This has a mix of MM/DD/YY and words
                    reference_numbers : list[str]
                '''
                entities = data["entities"]

                '''
                A short description of the document.
                '''
                additional_notes = data["additional_notes"]

                df.loc[len(df)] = {
                    "page_number": metadata["page_number"],
                    "document_number": metadata["document_number"],
                    "date": metadata["date"],
                    "document_type": metadata["document_type"],
                    "has_handwriting": metadata["has_handwriting"],
                    "has_stampts": metadata["has_stamps"],
                    "full_text": full_text,
                    "people": entities["people"],
                    "organizations": entities["organizations"],
                    "locations": entities["locations"],
                    "dates": entities["dates"],
                    "reference_numbers": entities["reference_numbers"],
                    "additional_notes": additional_notes
                }
            except Exception as e:
                print(file, e)
    
    df.to_parquet(f"{dt}.parquet")



./docs/IMAGES005/DOJ-OGR-00014722.json 'entities'
./docs/IMAGES005/DOJ-OGR-00013728.json 'entities'
./docs/IMAGES006/DOJ-OGR-00017360.json 'entities'
./docs/IMAGES001/DOJ-OGR-00001889.json 'entities'
./docs/IMAGES008/DOJ-OGR-00021980.json 'entities'
./docs/IMAGES009/DOJ-OGR-00023652.json 'entities'


# Distribution of Entities

## People

In [11]:
# Dictionary "DOCS" of general document type -> parquet file name
DOCS = {
    "court": "court_documents.parquet",
    "emails": "email_documents.parquet",
    "logs": "log_documents.parquet",
    "myspace": "myspace_documents.parquet",
    "reports": "report_documents.parquet"
}

In [ ]:
query = f"""
    WITH
        people_unnested AS (
            SELECT
                document_number,
                p AS person
            FROM
                '{DOCS["emails"]}' AS t,
                UNNEST(t.people) AS u(p)
        )
    SELECT
        person,
        COUNT(*) AS frequency
    FROM
        people_unnested
    -- WHERE
    --    not (person ILIKE '%epstein%')
    GROUP BY
        person
    ORDER BY
        frequency DESC
"""
email_df = duckdb.query(query)
# email_df.to_csv("./eda_imgs/people_emails.csv")

duckdb.query(query)